In [1]:
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
import kaiwu as kw
import sklearn
import itertools
import matplotlib.pyplot as plt
import sklearn.cluster
import time
import math
import warnings
import copy
warnings.filterwarnings("ignore")
import os
from sklearn.metrics import normalized_mutual_info_score
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score, v_measure_score

In [2]:
# In[2]:


# 加载真实的UCI家庭用电量数据集
def load_household_power_data():
    # UCI家庭用电量数据集URL
    uci_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00235/household_power_consumption.zip"
    save_path = "household_power_consumption.zip"
    data_file = "household_power_consumption.txt"
    
    # 检查是否已下载数据
    if not os.path.exists(data_file):
        print("下载UCI家庭用电量数据集...")
        try:
            import urllib.request
            urllib.request.urlretrieve(uci_url, save_path)
            
            # 解压文件
            import zipfile
            with zipfile.ZipFile(save_path, 'r') as zip_ref:
                zip_ref.extractall(".")
            print("数据集下载并解压完成")
        except:
            print("下载失败，使用备用数据源")
            # 如果下载失败，使用备用方法
            return load_backup_data()
    
    # 读取数据
    print("读取家庭用电量数据...")
    df = pd.read_csv(data_file, sep=';', low_memory=False)
    
    # 数据预处理
    # 合并日期和时间列
    df['DateTime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%d/%m/%Y %H:%M:%S')
    
    # 处理缺失值（用?表示）
    df = df.replace('?', np.nan)
    
    # 选择数值列并转换为float
    numeric_cols = ['Global_active_power', 'Global_reactive_power', 'Voltage', 
                   'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']
    
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # 按日期聚合，计算每日用电量统计
    df['Date'] = pd.to_datetime(df['Date'], format='%d/%m/%Y')
    daily_data = df.groupby('Date').agg({
        'Global_active_power': ['mean', 'sum', 'std'],
        'Global_reactive_power': 'mean',
        'Voltage': 'mean',
        'Global_intensity': 'mean',
        'Sub_metering_1': 'sum',
        'Sub_metering_2': 'sum',
        'Sub_metering_3': 'sum'
    }).reset_index()
    
    # 扁平化列名
    daily_data.columns = ['Date', 'Global_active_power_mean', 'Global_active_power_sum', 
                         'Global_active_power_std', 'Global_reactive_power_mean', 
                         'Voltage_mean', 'Global_intensity_mean', 'Sub_metering_1_sum',
                         'Sub_metering_2_sum', 'Sub_metering_3_sum']
    
    # 移除包含NaN的行
    daily_data = daily_data.dropna()
    
    # 创建目标变量：基于总用电量将日期分为高耗能和低耗能两类
    total_power_threshold = daily_data['Global_active_power_sum'].median()
    daily_data['label'] = (daily_data['Global_active_power_sum'] > total_power_threshold).astype(int)
    
    # 选择特征列
    feature_cols = ['Global_active_power_mean', 'Global_active_power_std', 
                   'Global_reactive_power_mean', 'Voltage_mean', 
                   'Global_intensity_mean', 'Sub_metering_1_sum',
                   'Sub_metering_2_sum', 'Sub_metering_3_sum']
    
    X_sample = daily_data[feature_cols].values
    y_sample = daily_data['label'].values
    
    # 由于数据可能很多，我们随机选择50个样本保持与之前相同的实验规模
    np.random.seed(66)
    if len(X_sample) > 50:
        indices = np.random.choice(len(X_sample), 50, replace=False)
        X_sample = X_sample[indices]
        y_sample = y_sample[indices]
    
    print("采样后各类别样本数量：", np.bincount(y_sample))
    print("原始数据形状：", X_sample.shape)
    print("特征列表：", feature_cols)
    
    # 数据归一化
    X_min = np.min(X_sample, 0, keepdims=True)
    X_max = np.max(X_sample, 0, keepdims=True)
    X_normalized = (X_sample - X_min) / (X_max - X_min)
    
    print("归一化后数据形状：", X_normalized.shape)
    print("使用的特征数量：", X_normalized.shape[1])
    
    return X_normalized, y_sample

def load_backup_data():
    """备用数据加载方法，如果主方法失败则使用"""
    print("使用备用数据加载方法...")
    # 这里可以使用其他方式获取数据，比如从其他URL或使用内置数据集
    # 作为最后手段，我们仍然使用模拟数据但更接近真实分布
    
    # 基于真实家庭用电量数据的统计特征生成更真实的模拟数据
    np.random.seed(66)
    n_samples = 50
    n_features = 8
    
    # 基于真实数据的统计特征生成两类数据
    # 低耗能模式（类别0）
    X_class0 = np.random.multivariate_normal(
        mean=[1.5, 0.8, 0.2, 240, 6, 2, 1, 3],
        cov=np.diag([0.5, 0.3, 0.1, 10, 2, 1, 0.5, 1]),
        size=25
    )
    
    # 高耗能模式（类别1）
    X_class1 = np.random.multivariate_normal(
        mean=[3.5, 1.5, 0.5, 242, 15, 8, 4, 10],
        cov=np.diag([1.0, 0.5, 0.2, 15, 4, 2, 1, 3]),
        size=25
    )
    
    X_sample = np.vstack([X_class0, X_class1])
    y_sample = np.hstack([np.zeros(25), np.ones(25)])
    
    print("采样后各类别样本数量：", np.bincount(y_sample.astype(int)))
    print("原始数据形状：", X_sample.shape)
    
    # 数据归一化
    X_min = np.min(X_sample, 0, keepdims=True)
    X_max = np.max(X_sample, 0, keepdims=True)
    X_normalized = (X_sample - X_min) / (X_max - X_min)
    
    print("归一化后数据形状：", X_normalized.shape)
    print("使用的特征数量：", X_normalized.shape[1])
    
    return X_normalized, y_sample

X_normalized, y_sample = load_household_power_data()

读取家庭用电量数据...
采样后各类别样本数量： [22 28]
原始数据形状： (50, 8)
特征列表： ['Global_active_power_mean', 'Global_active_power_std', 'Global_reactive_power_mean', 'Voltage_mean', 'Global_intensity_mean', 'Sub_metering_1_sum', 'Sub_metering_2_sum', 'Sub_metering_3_sum']
归一化后数据形状： (50, 8)
使用的特征数量： 8


In [3]:
clustering = sklearn.cluster.SpectralClustering(
    n_clusters=2,  # 2类：高耗能和低耗能
    gamma=1,
    affinity='rbf',
    assign_labels='kmeans'
).fit(X_normalized)
label_spectral = clustering.labels_
print(f"经典谱聚类标签: {label_spectral}")

# 计算真实标签与经典谱聚类的NMI
nmi_true_vs_spectral = normalized_mutual_info_score(y_sample, label_spectral)
print(f"真实标签 vs 经典谱聚类 NMI: {nmi_true_vs_spectral:.4f}")

经典谱聚类标签: [1 0 0 0 0 0 0 0 0 1 0 0 1 0 1 1 0 0 0 1 0 0 1 0 1 1 0 1 1 0 0 1 0 0 0 0 1
 1 0 1 0 0 0 0 0 0 0 0 0 0]
真实标签 vs 经典谱聚类 NMI: 0.3457


In [4]:
W_dist = cdist(X_normalized, X_normalized)
non_zero_dist = W_dist[W_dist != 0]
delta = np.median(non_zero_dist)  
W = np.exp(-np.power(W_dist / delta, 2))
np.fill_diagonal(W, 0)
D = np.diag(np.sum(W, axis=0)) 
mask = D != 0
D_inv = np.zeros_like(D)
D_inv[mask] = 1 / D[mask]
D_inv_sqrt = np.sqrt(D_inv)
L = D - W  
L = D_inv_sqrt @ L @ D_inv_sqrt  
cond_L = np.linalg.norm(L) 
print(f'cond_L = {cond_L}')


start_time = time.time()  # 使用time.time()函数 
eigenvalues, eigenvectors = np.linalg.eigh(L) 
end_time = time.time()  # 使用不同的变量名
print(f"特征分解耗时: {end_time - start_time:.4f}秒")


argsort_eigenvalues = np.argsort(eigenvalues)
idx = argsort_eigenvalues[1:3]  # 选取前2个特征向量
label_kmeans = sklearn.cluster.KMeans(
    n_clusters=2,  # 2类
    random_state=42,
    n_init=10  
).fit_predict(eigenvectors[:, idx])

# 计算相对于经典谱聚类的NMI
nmi_manual_vs_spectral = normalized_mutual_info_score(label_spectral, label_kmeans)
nmi_manual_vs_true = normalized_mutual_info_score(y_sample, label_kmeans)

print("选取的特征值索引：", idx)
print(f"手动谱聚类 vs 真实标签 NMI: {nmi_manual_vs_true:.4f}")
print(f"手动谱聚类 vs 经典谱聚类 NMI: {nmi_manual_vs_spectral:.4f}")

cond_L = 7.161177274282393
特征分解耗时: 0.0024秒
选取的特征值索引： [1 2]
手动谱聚类 vs 真实标签 NMI: 0.1365
手动谱聚类 vs 经典谱聚类 NMI: 0.7098


In [5]:
Lambda = np.linalg.norm(L,1)
numSamples = X_normalized.shape[0]
c = np.ones((numSamples,1))
c = c / np.linalg.norm(c,2)
I = np.eye(L.shape[0])
H0 = L - Lambda*I + Lambda*np.dot(c,c.T)
eigenvalues_H0, eigenvectors_H0 = np.linalg.eigh(H0)
print(Lambda)
print(np.max(eigenvalues))

2.154473569977809
1.1002124866129555


In [6]:
v = np.array([-0.2,-0.2,-0.05,0.1,0.2,0.2])
K = np.kron(I,v)
H = np.dot( K.T, np.dot(H0,K) )
cond_H = np.linalg.cond(H)
print(f'cond_H = {cond_H}')
H_min = np.min(H)
H_max = np.max(H)
print(H_min,H_max)
# 线性缩放矩阵
H_scaled = ((H - H_min) / (H_max - H_min)) * (127 - (-128)) + (-128)
#H_scaled = H/H_max * 127
# 四舍五入到整数
H_rounded = np.round(H_scaled)
# 确保数值在 -128 到 127 之间
H_clipped = np.clip(H_rounded, -128, 127)
H_qubo = kw.qubo.adjust_qubo_matrix_precision(H_clipped,bit_width=8)
print(H_qubo.shape)
print(H_qubo)

cond_H = 2.1626662182886778e+275
-0.04445536394313012 0.04445536394313012
(300, 300)
[[-440. -784. -192. ...  -16.  -16.  -16.]
 [  -0. -440. -192. ...  -16.  -16.  -16.]
 [  -0.   -0.  108. ...   -8.   -8.   -8.]
 ...
 [  -0.   -0.   -0. ...   28. -392. -392.]
 [  -0.   -0.   -0. ...   -0. -348. -784.]
 [  -0.   -0.   -0. ...   -0.   -0. -348.]]


In [7]:

qubo_model = kw.qubo.qubo_matrix_to_qubo_model(H_qubo)
H_ising = kw.qubo.qubo_matrix_to_ising_matrix(H_qubo)
ising_model = kw.qubo.qubo_model_to_ising_model(qubo_model)
print(H_ising[0].shape)
print(H_ising)


(301, 301)
(array([[ -0.,  98.,  24., ...,   2.,   2., 119.],
       [ 98.,  -0.,  24., ...,   2.,   2., 119.],
       [ 24.,  24.,  -0., ...,   1.,   1., 109.],
       ...,
       [  2.,   2.,   1., ...,  -0.,  98., 122.],
       [  2.,   2.,   1., ...,  98.,  -0., 122.],
       [119., 119., 109., ..., 122., 122.,  -0.]]), -52092.0)


In [8]:
kw.common.CheckpointManager.save_dir = '/tmp'
optimizer = kw.cim.CIMOptimizer(user_id="121839779389169666", sdk_code="A8cYsrBaetsTTYQGWsJwS5TUueFJ2X",task_name='household_power_01')  
solution_ising = optimizer.solve(H_ising[0]) 

2025-12-17 13:47:06,436 - kaiwu.common._logger - <module> - 3 - INFO - Task calculation successful!, Task name: household_power_01


In [9]:

solution_ising = optimizer.solve(H_ising[0])
solution = solution_ising[:,0:solution_ising.shape[1]-1]
delta = solution_ising[:,-1]
solution = np.multiply(solution,delta.reshape(-1,1))
solution = (solution + 1)/2
solution_realnumber = np.dot(K,solution.T)
print(solution.shape)
print(K.shape)

2025-12-17 13:47:06,502 - kaiwu.common._logger - <module> - 1 - INFO - Task calculation successful!, Task name: household_power_01
(10, 300)
(50, 300)


In [10]:
numSolution = solution_realnumber.shape[1]
loss = np.zeros(numSolution) - 1
label_quantum = np.zeros((numSolution,solution_realnumber.shape[0]))
for i in range(numSolution):
    h = solution_realnumber[:,i]
    h = h.reshape(-1,1)
    loss[i] = np.dot( h.T, np.dot(L,h) )
    h = (h - np.min(h))/( np.max(h) - np.min(h) )
    l = np.ones((solution_realnumber.shape[0],1))
    h = np.concatenate( (l,h),1 )
    label_quantum[i,:] = sklearn.cluster.KMeans(n_clusters=2).fit_predict(h)  # 2类
hamiltonian = kw.common.hamiltonian(H_ising[0], solution_ising)
print(hamiltonian)
print(label_quantum)

[-119640. -119636. -119632. -119632. -119628. -119608. -119588. -119548.
 -119516. -119512.]
[[0. 1. 0. 1. 1. 1. 1. 1. 0. 0. 1. 1. 0. 1. 0. 0. 1. 0. 1. 0. 1. 1. 0. 0.
  0. 0. 0. 0. 0. 1. 1. 0. 1. 1. 1. 1. 0. 0. 1. 0. 1. 1. 1. 0. 1. 1. 1. 0.
  0. 1.]
 [0. 1. 0. 1. 1. 1. 1. 1. 0. 0. 1. 1. 0. 1. 0. 0. 1. 0. 1. 0. 1. 1. 0. 0.
  0. 0. 0. 0. 0. 1. 1. 0. 1. 1. 1. 1. 0. 0. 1. 0. 1. 1. 1. 0. 1. 1. 1. 0.
  0. 1.]
 [0. 1. 0. 1. 1. 1. 1. 1. 0. 0. 1. 1. 0. 1. 0. 0. 1. 0. 1. 0. 1. 1. 0. 0.
  0. 0. 0. 0. 0. 1. 1. 0. 1. 1. 1. 1. 0. 0. 1. 0. 1. 1. 1. 0. 1. 1. 1. 0.
  0. 1.]
 [1. 0. 1. 0. 0. 0. 0. 0. 1. 1. 0. 0. 1. 0. 1. 1. 0. 1. 0. 1. 0. 0. 1. 1.
  1. 1. 1. 1. 1. 0. 0. 1. 0. 0. 0. 0. 1. 1. 0. 1. 0. 0. 0. 1. 0. 0. 0. 1.
  1. 0.]
 [1. 0. 1. 0. 0. 0. 0. 0. 1. 1. 0. 0. 1. 0. 1. 1. 0. 1. 0. 1. 0. 0. 1. 1.
  1. 1. 1. 1. 1. 0. 0. 1. 0. 0. 0. 0. 1. 1. 0. 1. 0. 0. 0. 1. 0. 0. 0. 1.
  1. 0.]
 [1. 0. 1. 0. 0. 0. 0. 0. 1. 1. 0. 0. 1. 0. 1. 1. 0. 1. 0. 1. 0. 0. 1. 1.
  1. 1. 1. 1. 1. 0. 0. 1. 0. 0. 0. 0. 1. 1. 0. 1

In [11]:
def purity_score(y_true, y_pred):
    """计算聚类纯度"""
    # 构建混淆矩阵
    contingency_matrix = sklearn.metrics.cluster.contingency_matrix(y_true, y_pred)
    # 对每个聚类，取最多数类别的样本数，求和
    return np.sum(np.amax(contingency_matrix, axis=0)) / np.sum(contingency_matrix)

In [12]:
# 计算所有评估指标
print("=" * 80)
print("家庭用电量数据聚类结果评估")
print("=" * 80)

# 1. 与真实标签比较
print("\n1. 与真实标签比较:")
print("-" * 40)

# 经典谱聚类 vs 真实标签
nmi_spectral_vs_true = normalized_mutual_info_score(y_sample, label_spectral)
ari_spectral_vs_true = adjusted_rand_score(y_sample, label_spectral)
v_measure_spectral_vs_true = v_measure_score(y_sample, label_spectral)
purity_spectral_vs_true = purity_score(y_sample, label_spectral)

print(f"经典谱聚类:")
print(f"  NMI:     {nmi_spectral_vs_true:.4f}")
print(f"  ARI:     {ari_spectral_vs_true:.4f}")
print(f"  V-measure: {v_measure_spectral_vs_true:.4f}")
print(f"  纯度:    {purity_spectral_vs_true:.4f}")

# 手动谱聚类 vs 真实标签
nmi_kmeans_vs_true = normalized_mutual_info_score(y_sample, label_kmeans)
ari_kmeans_vs_true = adjusted_rand_score(y_sample, label_kmeans)
v_measure_kmeans_vs_true = v_measure_score(y_sample, label_kmeans)
purity_kmeans_vs_true = purity_score(y_sample, label_kmeans)

print(f"\n手动谱聚类:")
print(f"  NMI:     {nmi_kmeans_vs_true:.4f}")
print(f"  ARI:     {ari_kmeans_vs_true:.4f}")
print(f"  V-measure: {v_measure_kmeans_vs_true:.4f}")
print(f"  纯度:    {purity_kmeans_vs_true:.4f}")

# 量子谱聚类 vs 真实标签（取第一个解）
nmi_quantum_vs_true = normalized_mutual_info_score(y_sample, label_quantum[0,:])
ari_quantum_vs_true = adjusted_rand_score(y_sample, label_quantum[0,:])
v_measure_quantum_vs_true = v_measure_score(y_sample, label_quantum[0,:])
purity_quantum_vs_true = purity_score(y_sample, label_quantum[0,:])

print(f"\n量子谱聚类:")
print(f"  NMI:     {nmi_quantum_vs_true:.4f}")
print(f"  ARI:     {ari_quantum_vs_true:.4f}")
print(f"  V-measure: {v_measure_quantum_vs_true:.4f}")
print(f"  纯度:    {purity_quantum_vs_true:.4f}")

# 2. 以经典谱聚类为基准比较
print("\n2. 以经典谱聚类为基准比较:")
print("-" * 40)

# 手动谱聚类 vs 经典谱聚类
nmi_kmeans_vs_spectral = normalized_mutual_info_score(label_spectral, label_kmeans)
ari_kmeans_vs_spectral = adjusted_rand_score(label_spectral, label_kmeans)
v_measure_kmeans_vs_spectral = v_measure_score(label_spectral, label_kmeans)
purity_kmeans_vs_spectral = purity_score(label_spectral, label_kmeans)

print(f"手动谱聚类 vs 经典谱聚类:")
print(f"  NMI:     {nmi_kmeans_vs_spectral:.4f}")
print(f"  ARI:     {ari_kmeans_vs_spectral:.4f}")
print(f"  V-measure: {v_measure_kmeans_vs_spectral:.4f}")
print(f"  纯度:    {purity_kmeans_vs_spectral:.4f}")

# 量子谱聚类 vs 经典谱聚类
nmi_quantum_vs_spectral = normalized_mutual_info_score(label_spectral, label_quantum[0,:])
ari_quantum_vs_spectral = adjusted_rand_score(label_spectral, label_quantum[0,:])
v_measure_quantum_vs_spectral = v_measure_score(label_spectral, label_quantum[0,:])
purity_quantum_vs_spectral = purity_score(label_spectral, label_quantum[0,:])

print(f"\n量子谱聚类 vs 经典谱聚类:")
print(f"  NMI:     {nmi_quantum_vs_spectral:.4f}")
print(f"  ARI:     {ari_quantum_vs_spectral:.4f}")
print(f"  V-measure: {v_measure_quantum_vs_spectral:.4f}")
print(f"  纯度:    {purity_quantum_vs_spectral:.4f}")

# 3. 以手动谱聚类为基准比较
print("\n3. 以手动谱聚类为基准比较:")
print("-" * 40)

# 经典谱聚类 vs 手动谱聚类
# 注意：这里的NMI、ARI、V-measure、纯度与上面的"手动谱聚类 vs 经典谱聚类"是相同的
# 因为这些指标都是对称的，但为了完整性我们还是计算一次
nmi_spectral_vs_kmeans = normalized_mutual_info_score(label_kmeans, label_spectral)
ari_spectral_vs_kmeans = adjusted_rand_score(label_kmeans, label_spectral)
v_measure_spectral_vs_kmeans = v_measure_score(label_kmeans, label_spectral)
purity_spectral_vs_kmeans = purity_score(label_kmeans, label_spectral)

print(f"经典谱聚类 vs 手动谱聚类:")
print(f"  NMI:     {nmi_spectral_vs_kmeans:.4f}")
print(f"  ARI:     {ari_spectral_vs_kmeans:.4f}")
print(f"  V-measure: {v_measure_spectral_vs_kmeans:.4f}")
print(f"  纯度:    {purity_spectral_vs_kmeans:.4f}")

# 量子谱聚类 vs 手动谱聚类
nmi_quantum_vs_kmeans = normalized_mutual_info_score(label_kmeans, label_quantum[0,:])
ari_quantum_vs_kmeans = adjusted_rand_score(label_kmeans, label_quantum[0,:])
v_measure_quantum_vs_kmeans = v_measure_score(label_kmeans, label_quantum[0,:])
purity_quantum_vs_kmeans = purity_score(label_kmeans, label_quantum[0,:])

print(f"\n量子谱聚类 vs 手动谱聚类:")
print(f"  NMI:     {nmi_quantum_vs_kmeans:.4f}")
print(f"  ARI:     {ari_quantum_vs_kmeans:.4f}")
print(f"  V-measure: {v_measure_quantum_vs_kmeans:.4f}")
print(f"  纯度:    {purity_quantum_vs_kmeans:.4f}")

# 4. 以量子谱聚类为基准比较
print("\n4. 以量子谱聚类为基准比较:")
print("-" * 40)

# 经典谱聚类 vs 量子谱聚类
# 注意：这些指标也是对称的
nmi_spectral_vs_quantum = normalized_mutual_info_score(label_quantum[0,:], label_spectral)
ari_spectral_vs_quantum = adjusted_rand_score(label_quantum[0,:], label_spectral)
v_measure_spectral_vs_quantum = v_measure_score(label_quantum[0,:], label_spectral)
purity_spectral_vs_quantum = purity_score(label_quantum[0,:], label_spectral)

print(f"经典谱聚类 vs 量子谱聚类:")
print(f"  NMI:     {nmi_spectral_vs_quantum:.4f}")
print(f"  ARI:     {ari_spectral_vs_quantum:.4f}")
print(f"  V-measure: {v_measure_spectral_vs_quantum:.4f}")
print(f"  纯度:    {purity_spectral_vs_quantum:.4f}")

# 手动谱聚类 vs 量子谱聚类
# 注意：这些指标也是对称的
nmi_kmeans_vs_quantum = normalized_mutual_info_score(label_quantum[0,:], label_kmeans)
ari_kmeans_vs_quantum = adjusted_rand_score(label_quantum[0,:], label_kmeans)
v_measure_kmeans_vs_quantum = v_measure_score(label_quantum[0,:], label_kmeans)
purity_kmeans_vs_quantum = purity_score(label_quantum[0,:], label_kmeans)

print(f"\n手动谱聚类 vs 量子谱聚类:")
print(f"  NMI:     {nmi_kmeans_vs_quantum:.4f}")
print(f"  ARI:     {ari_kmeans_vs_quantum:.4f}")
print(f"  V-measure: {v_measure_kmeans_vs_quantum:.4f}")
print(f"  纯度:    {purity_kmeans_vs_quantum:.4f}")

print("\n" + "=" * 80)

家庭用电量数据聚类结果评估

1. 与真实标签比较:
----------------------------------------
经典谱聚类:
  NMI:     0.3457
  ARI:     0.2153
  V-measure: 0.3457
  纯度:    0.7400

手动谱聚类:
  NMI:     0.1365
  ARI:     0.1121
  V-measure: 0.1365
  纯度:    0.6800

量子谱聚类:
  NMI:     0.3503
  ARI:     0.3975
  V-measure: 0.3503
  纯度:    0.8200

2. 以经典谱聚类为基准比较:
----------------------------------------
手动谱聚类 vs 经典谱聚类:
  NMI:     0.7098
  ARI:     0.7678
  V-measure: 0.7098
  纯度:    0.9400

量子谱聚类 vs 经典谱聚类:
  NMI:     0.4823
  ARI:     0.4525
  V-measure: 0.4823
  纯度:    0.8400

3. 以手动谱聚类为基准比较:
----------------------------------------
经典谱聚类 vs 手动谱聚类:
  NMI:     0.7098
  ARI:     0.7678
  V-measure: 0.7098
  纯度:    0.9400

量子谱聚类 vs 手动谱聚类:
  NMI:     0.4524
  ARI:     0.5090
  V-measure: 0.4524
  纯度:    0.8600

4. 以量子谱聚类为基准比较:
----------------------------------------
经典谱聚类 vs 量子谱聚类:
  NMI:     0.4823
  ARI:     0.4525
  V-measure: 0.4823
  纯度:    0.8400

手动谱聚类 vs 量子谱聚类:
  NMI:     0.4524
  ARI:     0.5090
  V-measure: 0.4524
  纯度

In [13]:
# 创建结果汇总表格
print("\n聚类结果汇总表")
print("=" * 100)
print(f"{'比较组':<38} {'NMI':<10} {'ARI':<8} {'V-measure':<10} {'Purity':<10}")
print("-" * 100)

# 第1组：与真实标签比较
print(f"{'[1] 经典谱聚类 vs 真实标签':<30} {nmi_spectral_vs_true:<10.4f} {ari_spectral_vs_true:<10.4f} {v_measure_spectral_vs_true:<10.4f} {purity_spectral_vs_true:<10.4f}")
print(f"{'[2] 手动谱聚类 vs 真实标签':<30} {nmi_kmeans_vs_true:<10.4f} {ari_kmeans_vs_true:<10.4f} {v_measure_kmeans_vs_true:<10.4f} {purity_kmeans_vs_true:<10.4f}")
print(f"{'[3] 量子谱聚类 vs 真实标签':<30} {nmi_quantum_vs_true:<10.4f} {ari_quantum_vs_true:<10.4f} {v_measure_quantum_vs_true:<10.4f} {purity_quantum_vs_true:<10.4f}")

print("-" * 100)

# 第2组：以经典谱聚类为基准
print(f"{'[4] 手动谱聚类 vs 经典谱聚类':<30} {nmi_kmeans_vs_spectral:<10.4f} {ari_kmeans_vs_spectral:<10.4f} {v_measure_kmeans_vs_spectral:<10.4f} {purity_kmeans_vs_spectral:<10.4f}")
print(f"{'[5] 量子谱聚类 vs 经典谱聚类':<30} {nmi_quantum_vs_spectral:<10.4f} {ari_quantum_vs_spectral:<10.4f} {v_measure_quantum_vs_spectral:<10.4f} {purity_quantum_vs_spectral:<10.4f}")

print("-" * 100)

# 第3组：以手动谱聚类为基准
print(f"{'[6] 经典谱聚类 vs 手动谱聚类':<30} {nmi_spectral_vs_kmeans:<10.4f} {ari_spectral_vs_kmeans:<10.4f} {v_measure_spectral_vs_kmeans:<10.4f} {purity_spectral_vs_kmeans:<10.4f}")
print(f"{'[7] 量子谱聚类 vs 手动谱聚类':<30} {nmi_quantum_vs_kmeans:<10.4f} {ari_quantum_vs_kmeans:<10.4f} {v_measure_quantum_vs_kmeans:<10.4f} {purity_quantum_vs_kmeans:<10.4f}")

print("-" * 100)

# 第4组：以量子谱聚类为基准
print(f"{'[8] 经典谱聚类 vs 量子谱聚类':<30} {nmi_spectral_vs_quantum:<10.4f} {ari_spectral_vs_quantum:<10.4f} {v_measure_spectral_vs_quantum:<10.4f} {purity_spectral_vs_quantum:<10.4f}")
print(f"{'[9] 手动谱聚类 vs 量子谱聚类':<30} {nmi_kmeans_vs_quantum:<10.4f} {ari_kmeans_vs_quantum:<10.4f} {v_measure_kmeans_vs_quantum:<10.4f} {purity_kmeans_vs_quantum:<10.4f}")

print("=" * 100)


聚类结果汇总表
比较组                                    NMI        ARI      V-measure  Purity    
----------------------------------------------------------------------------------------------------
[1] 经典谱聚类 vs 真实标签              0.3457     0.2153     0.3457     0.7400    
[2] 手动谱聚类 vs 真实标签              0.1365     0.1121     0.1365     0.6800    
[3] 量子谱聚类 vs 真实标签              0.3503     0.3975     0.3503     0.8200    
----------------------------------------------------------------------------------------------------
[4] 手动谱聚类 vs 经典谱聚类             0.7098     0.7678     0.7098     0.9400    
[5] 量子谱聚类 vs 经典谱聚类             0.4823     0.4525     0.4823     0.8400    
----------------------------------------------------------------------------------------------------
[6] 经典谱聚类 vs 手动谱聚类             0.7098     0.7678     0.7098     0.9400    
[7] 量子谱聚类 vs 手动谱聚类             0.4524     0.5090     0.4524     0.8600    
----------------------------------------------------------------------------------